# `basic_cells.ipynb` — cell library for MOEX × Chronos-2 runs

**This notebook is not meant to be executed top-to-bottom in production.** It's the canonical source of reusable cells. When you build `runner.ipynb` (or any stage notebook), you copy/import the relevant cells from here.

Sections:
1. Imports & GPU
2. Config loader (YAML)
3. Drive mount + cache paths
4. ISS soft prefetcher (rate-limited, retry, idempotent, multi-stage aware)
5. Cache-only loaders (no API fall-through during experiments)
6. Panel build + covariates + calendar features
7. Chronos input builder (long form, future_df)
8. Walk-forward driver
9. Metrics (DA + binomial + Wilson CI, corr, amplitude, coverage)
10. Baselines (B0/B1/B2/B3)
11. Plot helpers
12. Stage orchestrator (top-level `run_stage(cfg)` glue)

Each section is one or two cells. Copy what `runner.ipynb` needs; do not edit logic here in-place — propagate fixes from `runner.ipynb` back here when they stabilise.

## 1. Imports & GPU

In [ ]:
!pip install -q chronos-forecasting "pandas[pyarrow]" requests matplotlib numpy tqdm pyyaml scipy
# Path B (uncomment when running fine-tune stages):
# !pip install -q "autogluon.timeseries[chronos]"


In [ ]:
import os, math, json, time, warnings, hashlib, random
from dataclasses import dataclass, field, asdict
from pathlib import Path
from typing import Optional, Sequence, Any

import numpy as np
import pandas as pd
import requests
import yaml
import torch
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from scipy import stats as sstats

warnings.filterwarnings("ignore", category=FutureWarning)

HAS_CUDA = torch.cuda.is_available()
DEVICE = "cuda" if HAS_CUDA else "cpu"
if HAS_CUDA:
    cap = torch.cuda.get_device_capability(0)
    DTYPE = torch.bfloat16 if cap[0] >= 8 else torch.float16
    print(f"GPU: {torch.cuda.get_device_name(0)}  cap={cap}  dtype={DTYPE}")
else:
    DTYPE = torch.float32
    print("No CUDA — CPU only (slow).")


## 2. Config loader

YAML config with `.yaml` extension. The loader returns a plain `dict`; downstream code reads keys directly so config schema can evolve without breaking dataclass migrations. Required top-level keys are validated up front.

In [ ]:
REQUIRED_KEYS = [
    "stage_id", "interval", "tickers", "date_from", "date_till",
    "context_len", "horizon", "walk_forward", "covariates", "output_dir",
]

def load_config(path: str) -> dict:
    with open(path, "r", encoding="utf-8") as f:
        cfg = yaml.safe_load(f)
    missing = [k for k in REQUIRED_KEYS if k not in cfg]
    if missing:
        raise KeyError(f"config {path} missing required keys: {missing}")
    assert cfg["context_len"] <= 8192, "Chronos-2 max context is 8192"
    assert cfg["horizon"] <= 1024, "Chronos-2 max prediction_length is 1024"
    assert cfg["interval"] in (10, 60, 24), "interval must be 10, 60, or 24 (ISS has no 15m candle)"
    cfg.setdefault("indexes", ["IMOEX", "MOEXOG", "MOEXMM", "MOEXFN", "RGBI"])
    cfg.setdefault("futures_proxies", [])
    cfg.setdefault("quantiles", [0.1, 0.5, 0.9])
    cfg.setdefault("eval_horizons", [1, 2, 3, 5])
    cfg.setdefault("primary_horizons", [2, 3, 5])
    cfg.setdefault("baselines", ["zero", "last", "momentum5", "ar1"])
    return cfg

def save_config_snapshot(cfg: dict, out_dir: str):
    Path(out_dir).mkdir(parents=True, exist_ok=True)
    with open(Path(out_dir) / "config.yaml", "w", encoding="utf-8") as f:
        yaml.safe_dump(cfg, f, sort_keys=False)

## 3. Drive mount + cache paths

Cache lives on Drive so it survives runtime resets. The same `cache_dir` is shared across stages — files are keyed by (secid, engine, interval, date_from, date_till) so a daily 2021–2026 SBER pull is reused by every stage that needs it.

In [ ]:
def mount_drive(mount_at: str = "/content/drive") -> bool:
    try:
        from google.colab import drive
        if not os.path.ismount(mount_at):
            drive.mount(mount_at, force_remount=False)
        return True
    except Exception as e:
        print(f"Not in Colab or mount failed ({e}); falling back to local cache.")
        return False

def resolve_cache_dir(cfg: dict) -> str:
    # Priority: explicit cfg.cache_dir → Drive default → local ./moex_cache
    if "cache_dir" in cfg and cfg["cache_dir"]:
        path = cfg["cache_dir"]
    elif mount_drive():
        path = "/content/drive/MyDrive/moex_cache"
    else:
        path = "./moex_cache"
    Path(path).mkdir(parents=True, exist_ok=True)
    print(f"cache_dir = {path}")
    return path


## 4. ISS soft prefetcher

Goals:
- **Soft on ISS**: ≥ 0.25 s between requests, retry on 429/5xx with exponential backoff, never more than 3 retries.
- **Idempotent**: skip files already on disk. Re-run safe.
- **Multi-stage aware**: takes a *list* of (engine, market, secid, interval, date_from, date_till) tuples and walks them all. The runner builds that list from the union of every stage's config.
- **No silent failures**: every miss is logged with the exact URL and HTTP status.

The cache key includes engine and market so identical SECIDs across engines don't collide.

In [ ]:
ISS_BASE = "https://iss.moex.com/iss"
RATE_LIMIT_SEC = 0.25
RETRY_BACKOFF = (1, 3, 8)  # seconds

def _cache_key(engine: str, market: str, secid: str, interval: int, dfrom: str, dtill: str) -> str:
    return f"{engine}_{market}_{secid}_{interval}_{dfrom}_{dtill}.parquet"

def _iss_candles(engine: str, market: str, secid: str, dfrom: str, dtill: str, interval: int) -> pd.DataFrame:
    url = f"{ISS_BASE}/engines/{engine}/markets/{market}/securities/{secid}/candles.json"
    rows, cols, start = [], None, 0
    while True:
        params = {"from": dfrom, "till": dtill, "interval": interval, "start": start}
        for attempt, wait in enumerate([0] + list(RETRY_BACKOFF)):
            if wait: time.sleep(wait)
            try:
                r = requests.get(url, params=params, timeout=30)
                if r.status_code in (429, 500, 502, 503, 504):
                    continue
                r.raise_for_status()
                break
            except requests.RequestException as e:
                if attempt == len(RETRY_BACKOFF):
                    print(f"  FAIL {secid}: {e}")
                    return pd.DataFrame()
        time.sleep(RATE_LIMIT_SEC)
        data = r.json().get("candles", {"columns": [], "data": []})
        cols = data["columns"]
        chunk = data["data"]
        if not chunk: break
        rows.extend(chunk)
        if len(chunk) < 500: break
        start += len(chunk)
    df = pd.DataFrame(rows, columns=cols) if rows else pd.DataFrame(columns=cols or ["begin"])
    if not df.empty:
        df["begin"] = pd.to_datetime(df["begin"])
        df = df.sort_values("begin").reset_index(drop=True)
    return df

def prefetch_one(engine, market, secid, interval, dfrom, dtill, cache_dir, force=False) -> pd.DataFrame:
    fname = Path(cache_dir) / _cache_key(engine, market, secid, interval, dfrom, dtill)
    if fname.exists() and not force:
        print(f"{fname} already exist")
        return pd.read_parquet(fname)
    df = _iss_candles(engine, market, secid, dfrom, dtill, interval)
    if not df.empty:
        df.to_parquet(fname)
    else:
        print(f"  EMPTY {engine}/{market}/{secid} interval={interval} [{dfrom}..{dtill}]")
    return df

def build_prefetch_manifest(cfgs: list[dict]) -> list[tuple]:
    """Union the needs of every stage config into a deduped manifest."""
    seen, out = set(), []
    for cfg in cfgs:
        interval = cfg["interval"]; dfrom = cfg["date_from"]; dtill = cfg["date_till"]
        for t in cfg["tickers"]:
            key = ("stock", "shares", t, interval, dfrom, dtill)
            if key not in seen: seen.add(key); out.append(key)
        for idx in cfg.get("indexes", []):
            key = ("stock", "index", idx, interval, dfrom, dtill)
            if key not in seen: seen.add(key); out.append(key)
        for fut_root in cfg.get("futures_proxies", []):
            # FORTS root → expand to all candidate SECIDs; ISS serves FORTS at the
            # same intervals as equities (10m/60m/1d), so fetch at the target interval.
            source_interval = interval
            for secid in _enumerate_contracts(fut_root, dfrom, dtill):
                key = ("futures", "forts", secid, source_interval, dfrom, dtill)
                if key not in seen: seen.add(key); out.append(key)
    return out

def prefetch_all(manifest: list[tuple], cache_dir: str):
    """Walk the manifest; resumes mid-list because each item is independently cached."""
    for engine, market, secid, interval, dfrom, dtill in tqdm(manifest, desc="prefetch"):
        prefetch_one(engine, market, secid, interval, dfrom, dtill, cache_dir)

## 4b. FORTS chain resolver

FORTS contracts roll on a fixed cadence (Brent monthly; Si and Gold quarterly
H/M/U/Z). To build a continuous covariate series the resolver (a) enumerates
all candidate contracts whose lifetime could overlap the requested window,
(b) pulls per-contract candles via `prefetch_one` (reuses the soft prefetcher
+ cache), (c) picks the front-month contract per bar by daily volume, (d)
emits a stitched DataFrame carrying `contract_id` so the covariate builder
can NaN the return at every roll boundary.

ISS serves FORTS candles at the same intervals as equities (10m / 60m / 1d),
so each contract is fetched directly at the target interval — no cross-interval
fallback. (Note: ISS has no 15m candle for any market; the intraday stage uses
10m, see `exp_plan.md` §3 Stage 3.)

Resolved output is cached separately from per-contract candles so a re-run
skips both layers.

In [ ]:
FORTS_LETTER_MAP = {
    "BR": "FGHJKMNQUVXZ",  # Brent — monthly, all 12 letters
    "Si": "HMUZ",          # USD/RUB — quarterly H/M/U/Z
    "GD": "HMUZ",          # Gold   — quarterly H/M/U/Z
}

def _enumerate_contracts(root: str, dfrom: str, dtill: str) -> list[str]:
    """Candidate SECIDs whose lifetime could overlap [dfrom, dtill].
    Si trades ~2y before expiry → backward buffer 2y; +1y forward.
    Year-digit collisions are only across decades; 2019..2027 stays unique.
    """
    if root not in FORTS_LETTER_MAP:
        raise ValueError(f"unknown FORTS root: {root!r}; expected {list(FORTS_LETTER_MAP)}")
    letters = FORTS_LETTER_MAP[root]
    y_from = pd.to_datetime(dfrom).year - 2
    y_till = pd.to_datetime(dtill).year + 1
    return [f"{root}{L}{str(y % 10)}"
            for y in range(y_from, y_till + 1)
            for L in letters]

def _resolve_futures_chain(root: str, interval: int, dfrom: str, dtill: str,
                            cache_dir: str) -> pd.DataFrame:
    """Continuous FORTS series for `root` over [dfrom, dtill].
    Output schema: [begin, close, volume, contract_id] at the target interval
    (ISS serves FORTS at 10m/60m/1d, same as equities). Covariate builder
    regrids close onto the target interval, carries contract_id along, and NaNs
    the return at every contract switch.
    Cached at: {root}_forts_resolved_{interval}_{dfrom}_{dtill}.parquet.
    """
    cache_fname = Path(cache_dir) / f"{root}_forts_resolved_{interval}_{dfrom}_{dtill}.parquet"
    if cache_fname.exists():
        return pd.read_parquet(cache_fname)

    source_interval = interval
    parts = []
    for secid in _enumerate_contracts(root, dfrom, dtill):
        df = prefetch_one("futures", "forts", secid, source_interval, dfrom, dtill, cache_dir)
        if df.empty: continue
        parts.append(df[["begin", "close", "volume"]].assign(contract_id=secid))
    if not parts:
        raise RuntimeError(f"_resolve_futures_chain: no contracts found for "
                           f"root={root!r} interval={source_interval} {dfrom}..{dtill}")

    all_rows = pd.concat(parts, ignore_index=True)
    # Front-month per bar: highest volume wins; deterministic tiebreak via stable sort.
    all_rows = all_rows.sort_values(["begin", "volume"], ascending=[True, False], kind="stable")
    resolved = (all_rows.drop_duplicates(subset=["begin"], keep="first")
                        .sort_values("begin").reset_index(drop=True))
    resolved["begin"] = pd.to_datetime(resolved["begin"])

    # Validation: no intra-contract |log-return| > 0.20.
    close = resolved["close"].astype(float)
    ret_check = np.log(close / close.shift(1))
    boundary = resolved["contract_id"] != resolved["contract_id"].shift(1)
    intra = ret_check[~boundary]
    spikes = intra[intra.abs() > 0.20]
    if len(spikes):
        print(f"  WARN {root}: {len(spikes)} intra-contract bars with |log_return|>0.20")
        print(resolved.loc[spikes.index].head(5)[["begin", "contract_id", "close"]].to_string())

    resolved.to_parquet(cache_fname)
    return resolved

## 5. Cache-only loaders

During experiment runs we never hit the ISS API. If a file is missing it's a config bug — we fail loudly with the exact key. Production / live forecasting flips `allow_api_fallback=True`.

In [ ]:
class CacheMiss(KeyError):
    pass

def load_cached(engine, market, secid, interval, dfrom, dtill, cache_dir, allow_api_fallback=False) -> pd.DataFrame:
    fname = Path(cache_dir) / _cache_key(engine, market, secid, interval, dfrom, dtill)
    if fname.exists():
        return pd.read_parquet(fname)
    if allow_api_fallback:
        print(f"  cache miss → API: {fname.name}")
        return prefetch_one(engine, market, secid, interval, dfrom, dtill, cache_dir)
    raise CacheMiss(f"missing cache file: {fname}")

def load_stage_inputs(cfg: dict, cache_dir: str) -> tuple[dict, dict, dict]:
    interval, dfrom, dtill = cfg["interval"], cfg["date_from"], cfg["date_till"]
    prices = {t: load_cached("stock","shares",t,interval,dfrom,dtill,cache_dir) for t in cfg["tickers"]}
    indexes = {}
    for idx in cfg.get("indexes", []):
        try:
            df = load_cached("stock","index",idx,interval,dfrom,dtill,cache_dir)
            if not df.empty: indexes[idx] = df
        except CacheMiss as e:
            print(f"  skipping index {idx}: {e}")
    futures = {}
    for fut_root in cfg.get("futures_proxies", []):
        # FORTS root → continuous chain (see section 4b resolver).
        try:
            df = _resolve_futures_chain(fut_root, interval, dfrom, dtill, cache_dir)
            if not df.empty: futures[fut_root] = df
        except Exception as e:
            print(f"  skipping future {fut_root}: {e}")
    return prices, indexes, futures


## 6. Panel build + covariates + calendar features

Regularise per series, then inner-join across tickers so the panel index is identical for all ids (group attention requirement). Covariates are broadcast across all ids; calendar features only.

In [ ]:
def _session_grid(start, end, interval_min):
    days = pd.bdate_range(start.normalize(), end.normalize())
    bars_per_day = (8*60 + 50) // interval_min  # 10:00..18:50 MSK
    out = [pd.date_range(d + pd.Timedelta(hours=10), periods=bars_per_day, freq=f"{interval_min}min") for d in days]
    return pd.DatetimeIndex(np.concatenate(out)) if out else pd.DatetimeIndex([])

def to_regular_series(df, value_col, name, interval):
    if df.empty: return pd.Series(name=name, dtype=float)
    s = df.set_index("begin")[value_col].astype(float).sort_index()
    if interval == 24:
        s = s.asfreq("B")
    else:
        s = s.reindex(_session_grid(s.index.min(), s.index.max(), interval))
    return s.ffill().rename(name)

def build_price_panel(prices, interval):
    return pd.concat([to_regular_series(df,"close",t,interval) for t,df in prices.items()], axis=1).dropna(how="any")

def log_returns(panel):
    return np.log(panel / panel.shift(1)).dropna(how="any")

def calendar_features(index, interval):
    df = pd.DataFrame({
        "hour":  index.hour.astype(np.float32),
        "dow":   index.dayofweek.astype(np.float32),
        "dom":   index.day.astype(np.float32),
        "month": index.month.astype(np.float32),
    }, index=index)
    if interval != 24:
        df["session_open"] = ((index.hour >= 10) & (index.hour < 19)).astype(np.float32)
    return df

def build_covariate_panel(prices, indexes, futures, interval, ret_index):
    parts = []
    for name, df in indexes.items():
        s = to_regular_series(df,"close",f"{name}_close",interval)
        parts.append(np.log(s/s.shift(1)).rename(f"{name}_ret"))
    for name, df in futures.items():
        # Resolved chain: regrid close (ffill), carry contract_id, compute return
        # on the target interval, NaN at every roll boundary.
        s_close = to_regular_series(df, "close", f"{name}_close", interval)
        cid_src = df.set_index("begin")["contract_id"]
        s_cid = cid_src.reindex(s_close.index, method="ffill")
        ret = np.log(s_close / s_close.shift(1))
        boundary = (s_cid != s_cid.shift(1)).fillna(False)
        ret = ret.mask(boundary)
        parts.append(ret.rename(f"{name}_ret"))
    for tic, df in prices.items():
        v = to_regular_series(df,"volume",f"{tic}_vol",interval)
        parts.append(np.log1p(v).diff().rename(f"{tic}_dlogvol"))
    cov = pd.concat(parts, axis=1).reindex(ret_index).ffill().dropna(how="any")
    return cov

def assemble_panels(cfg, prices, indexes, futures):
    price = build_price_panel(prices, cfg["interval"])
    ret = log_returns(price)
    cov_mode = cfg["covariates"]  # "full", "calendar_only", "market_only", "none"
    if cov_mode == "none":
        cov = pd.DataFrame(index=ret.index)
    else:
        cov = build_covariate_panel(prices, indexes, futures, cfg["interval"], ret.index)
        if cov_mode == "calendar_only":
            cov = cov.iloc[:, 0:0]  # empty market cov; calendar handled separately
    # realign
    common = ret.index.intersection(cov.index) if not cov.empty else ret.index
    return price.loc[common], ret.loc[common], cov.loc[common]


## 7. Chronos input builder

In [ ]:
def build_chronos_inputs(ret_panel, cov_panel, tickers, context_len, horizon, t_anchor,
                          interval, covariate_mode="full"):
    """
    t_anchor = first index of the forecast horizon (so context ends at t_anchor-1).
    Returns (context_df, future_df, fut_idx).
    """
    pos = ret_panel.index.get_loc(t_anchor)
    ctx_idx = ret_panel.index[pos-context_len:pos]
    fut_idx = ret_panel.index[pos:pos+horizon]

    use_market_cov = covariate_mode in ("full", "market_only") and not cov_panel.empty
    use_calendar = covariate_mode in ("full", "calendar_only")
    cal_ctx = calendar_features(ctx_idx, interval) if use_calendar else None
    cal_fut = calendar_features(fut_idx, interval) if use_calendar else None
    cov_ctx = cov_panel.loc[ctx_idx] if use_market_cov else None

    ctx_rows, fut_rows = [], []
    for tic in tickers:
        block = pd.DataFrame({"id": tic, "timestamp": ctx_idx, "target": ret_panel[tic].loc[ctx_idx].values})
        if use_market_cov:
            for c in cov_ctx.columns: block[c] = cov_ctx[c].values
        if use_calendar:
            for c in cal_ctx.columns: block[c] = cal_ctx[c].values
        ctx_rows.append(block)
        fb = pd.DataFrame({"id": tic, "timestamp": fut_idx})
        if use_calendar:
            for c in cal_fut.columns: fb[c] = cal_fut[c].values
        fut_rows.append(fb)
    return pd.concat(ctx_rows, ignore_index=True), pd.concat(fut_rows, ignore_index=True), fut_idx


## 8. Walk-forward driver

Loops over forecast windows, calls Chronos once per window (group attention across all tickers), persists predictions to Parquet. Defensive on indexing — `t_anchor` must be far enough from both edges of the panel.

In [ ]:
def walk_forward_anchors(ret_index, context_len, horizon, shift, max_windows=None):
    starts = []
    pos = context_len
    end_pos = len(ret_index) - horizon
    while pos < end_pos:
        starts.append(ret_index[pos])
        pos += shift
    if max_windows and len(starts) > max_windows:
        # keep evenly spaced subsample if too many
        idx = np.linspace(0, len(starts)-1, max_windows).astype(int)
        starts = [starts[i] for i in idx]
    return starts

def run_walk_forward(pipeline, cfg, ret_panel, cov_panel, out_dir):
    preds_dir = Path(out_dir) / "preds"; preds_dir.mkdir(parents=True, exist_ok=True)
    wf = cfg["walk_forward"]
    anchors = walk_forward_anchors(
        ret_panel.index, cfg["context_len"], cfg["horizon"],
        shift=wf["shift"], max_windows=wf.get("max_windows"),
    )
    print(f"walk-forward: {len(anchors)} windows")
    all_preds = []
    for i, t_anchor in enumerate(tqdm(anchors, desc="windows")):
        ctx, fut, fut_idx = build_chronos_inputs(
            ret_panel, cov_panel, cfg["tickers"], cfg["context_len"], cfg["horizon"],
            t_anchor, cfg["interval"], cfg["covariates"],
        )
        try:
            pred = pipeline.predict_df(
                ctx, future_df=fut, prediction_length=cfg["horizon"],
                quantile_levels=list(cfg["quantiles"]),
                id_column="id", timestamp_column="timestamp", target="target",
            )
        except Exception as e:
            print(f"  window {i} ({t_anchor}) failed: {e}")
            continue
        pred["window"] = i
        pred["t_anchor"] = t_anchor
        all_preds.append(pred)
        if (i+1) % 25 == 0:  # checkpoint flush
            pd.concat(all_preds, ignore_index=True).to_parquet(preds_dir / "preds_partial.parquet")
    full = pd.concat(all_preds, ignore_index=True) if all_preds else pd.DataFrame()
    full.to_parquet(preds_dir / "preds.parquet")
    return full


## 9. Metrics

Per (ticker, horizon-step) we compute DA + binomial p + Wilson CI, Pearson r, Spearman ρ, |pred|/|true| median, q-coverage. Then bootstrap aggregates across tickers.

In [ ]:
def wilson_ci(k, n, z=1.96):
    if n == 0: return (np.nan, np.nan)
    p = k/n
    denom = 1 + z*z/n
    center = (p + z*z/(2*n))/denom
    half = z*math.sqrt(p*(1-p)/n + z*z/(4*n*n))/denom
    return (center-half, center+half)

def per_cell_metrics(pred_df, ret_panel, tickers, horizons, quantiles):
    median_q = str(quantiles[len(quantiles)//2])
    lo_q, hi_q = str(quantiles[0]), str(quantiles[-1])
    rows = []
    for tic in tickers:
        sub = pred_df[pred_df["id"] == tic].copy()
        if sub.empty: continue
        # per window: which horizon step is this row? step = 1..H within window
        sub = sub.sort_values(["window", "timestamp"])
        sub["h_step"] = sub.groupby("window").cumcount() + 1
        for h in horizons:
            slab = sub[sub["h_step"] == h]
            if slab.empty: continue
            y_pred = slab[median_q].values
            ts = slab["timestamp"].values
            try:
                y_true = ret_panel[tic].loc[pd.DatetimeIndex(ts)].values
            except KeyError:
                continue
            mask = ~(np.isnan(y_pred) | np.isnan(y_true))
            y_pred, y_true = y_pred[mask], y_true[mask]
            n = len(y_true)
            if n < 5: continue
            hits = int((np.sign(y_pred) == np.sign(y_true)).sum())
            da = hits / n
            p_binom = sstats.binomtest(hits, n, 0.5, alternative="greater").pvalue
            ci_lo, ci_hi = wilson_ci(hits, n)
            r_p = float(np.corrcoef(y_pred, y_true)[0,1]) if n > 1 else np.nan
            r_s = float(sstats.spearmanr(y_pred, y_true).correlation) if n > 1 else np.nan
            amp = float(np.median(np.abs(y_pred) / (np.abs(y_true) + 1e-12)))
            q_lo = slab[lo_q].values[mask]
            q_hi = slab[hi_q].values[mask]
            cov = float(((y_true >= q_lo) & (y_true <= q_hi)).mean())
            mae = float(np.mean(np.abs(y_true - y_pred)))
            rows.append(dict(ticker=tic, horizon=h, n=n, da=da, p_binom=p_binom,
                             da_ci_lo=ci_lo, da_ci_hi=ci_hi, pearson=r_p, spearman=r_s,
                             amp_ratio=amp, coverage=cov, mae=mae))
    out = pd.DataFrame(rows)
    if not out.empty:
        # BH-adjusted q across the (ticker, horizon) cells
        m = len(out)
        order = out["p_binom"].rank(method="first").astype(int).values
        out["p_binom_bh"] = np.minimum.accumulate((out["p_binom"].sort_values().values * m / np.arange(1, m+1))[::-1])[::-1][order-1]
    return out

def aggregate_metrics(per_cell):
    if per_cell.empty: return pd.DataFrame()
    agg = per_cell.groupby("horizon").agg(
        mean_da=("da","mean"), median_da=("da","median"),
        mean_pearson=("pearson","mean"), mean_spearman=("spearman","mean"),
        mean_amp=("amp_ratio","mean"), mean_cov=("coverage","mean"),
        n_cells=("ticker","count"),
        n_signif_05=("p_binom_bh", lambda s: int((s<0.05).sum())),
    ).reset_index()
    return agg


## 10. Baselines

Computed on the same walk-forward windows. The runner persists their predictions in the same shape as Chronos's so the metric function reuses the exact same code path.

In [ ]:
def baseline_predictions(name, ret_panel, tickers, anchors, context_len, horizon):
    rows = []
    for i, t in enumerate(anchors):
        pos = ret_panel.index.get_loc(t)
        for tic in tickers:
            ctx = ret_panel[tic].iloc[pos-context_len:pos].values
            fut_ts = ret_panel.index[pos:pos+horizon]
            if name == "zero":
                yhat = np.zeros(horizon)
            elif name == "last":
                yhat = np.full(horizon, ctx[-1] if len(ctx) else 0.0)
            elif name == "momentum5":
                yhat = np.full(horizon, np.nanmean(ctx[-5:]) if len(ctx)>=5 else 0.0)
            elif name == "ar1":
                if len(ctx) >= 30:
                    x, y = ctx[:-1], ctx[1:]
                    a = np.dot(x, y) / (np.dot(x, x) + 1e-12)
                    yhat = np.empty(horizon); last = ctx[-1]
                    for k in range(horizon):
                        last = a * last; yhat[k] = last
                else:
                    yhat = np.zeros(horizon)
            else:
                raise ValueError(name)
            for k, ts in enumerate(fut_ts):
                rows.append(dict(id=tic, timestamp=ts, window=i, t_anchor=t, **{"0.5": yhat[k], "0.1": yhat[k], "0.9": yhat[k]}))
    return pd.DataFrame(rows)


## 11. Plot helpers

In [ ]:
def plot_da_heatmap(per_cell, out_path, title=""):
    pivot = per_cell.pivot(index="ticker", columns="horizon", values="da")
    fig, ax = plt.subplots(figsize=(1.2*len(pivot.columns)+2, 0.4*len(pivot.index)+2))
    im = ax.imshow(pivot.values, cmap="RdYlGn", vmin=0.40, vmax=0.60, aspect="auto")
    ax.set_xticks(range(len(pivot.columns))); ax.set_xticklabels(pivot.columns)
    ax.set_yticks(range(len(pivot.index))); ax.set_yticklabels(pivot.index)
    for i in range(pivot.shape[0]):
        for j in range(pivot.shape[1]):
            v = pivot.values[i,j]
            ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=8)
    plt.colorbar(im, ax=ax, label="Dir. Acc.")
    ax.set_title(title or "Directional accuracy (ticker × horizon)")
    plt.tight_layout(); plt.savefig(out_path, dpi=120); plt.close()

def plot_da_vs_baseline(per_cell, per_cell_baseline, baseline_name, out_path):
    m = per_cell.merge(per_cell_baseline, on=["ticker","horizon"], suffixes=("","_b"))
    m["delta"] = m["da"] - m["da_b"]
    fig, ax = plt.subplots(figsize=(10, 4))
    for h, sub in m.groupby("horizon"):
        ax.bar(sub["ticker"] + f"|h={h}", sub["delta"], label=f"h={h}")
    ax.axhline(0, color="black", lw=0.7)
    ax.set_ylabel(f"DA(Chronos) - DA({baseline_name})")
    ax.tick_params(axis="x", rotation=80, labelsize=7)
    ax.legend(fontsize=8); plt.tight_layout(); plt.savefig(out_path, dpi=120); plt.close()

def plot_corr_hist(pred_df, ret_panel, horizons, quantiles, out_path):
    median_q = str(quantiles[len(quantiles)//2])
    fig, axes = plt.subplots(1, len(horizons), figsize=(4*len(horizons), 3), sharey=True)
    if len(horizons) == 1: axes = [axes]
    pred_df = pred_df.sort_values(["window","id","timestamp"])
    pred_df["h_step"] = pred_df.groupby(["window","id"]).cumcount() + 1
    for ax, h in zip(axes, horizons):
        rs = []
        slab = pred_df[pred_df["h_step"]==h]
        for (w, _id), s in slab.groupby(["window","id"]):
            y_p = s[median_q].values
            try: y_t = ret_panel[_id].loc[s["timestamp"].values].values
            except KeyError: continue
            if len(y_t) > 1 and not np.isnan(y_p).any():
                rs.append(np.corrcoef(y_p, y_t)[0,1])
        ax.hist(rs, bins=30); ax.axvline(0, color="red", lw=0.8)
        ax.set_title(f"h={h}, n={len(rs)}"); ax.set_xlabel("Pearson r")
    plt.suptitle("Per-window correlation (pred vs true), by horizon")
    plt.tight_layout(); plt.savefig(out_path, dpi=120); plt.close()

def plot_amplitude(per_cell, out_path):
    fig, ax = plt.subplots(figsize=(8,4))
    for h, sub in per_cell.groupby("horizon"):
        ax.scatter([h]*len(sub), sub["amp_ratio"], label=f"h={h}", alpha=0.6)
    ax.axhline(1.0, color="black", lw=0.6, ls="--")
    ax.set_xlabel("horizon"); ax.set_ylabel("median |pred|/|true|")
    ax.set_yscale("log"); ax.set_title("Amplitude calibration (target = 1.0)")
    plt.tight_layout(); plt.savefig(out_path, dpi=120); plt.close()

def plot_coverage(per_cell, out_path, target=0.80):
    pivot = per_cell.pivot(index="ticker", columns="horizon", values="coverage")
    fig, ax = plt.subplots(figsize=(1.2*len(pivot.columns)+2, 0.4*len(pivot.index)+2))
    im = ax.imshow(pivot.values, cmap="coolwarm", vmin=0.60, vmax=1.0, aspect="auto")
    ax.set_xticks(range(len(pivot.columns))); ax.set_xticklabels(pivot.columns)
    ax.set_yticks(range(len(pivot.index))); ax.set_yticklabels(pivot.index)
    for i in range(pivot.shape[0]):
        for j in range(pivot.shape[1]):
            v = pivot.values[i,j]
            ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=8)
    plt.colorbar(im, ax=ax, label=f"q-coverage (target={target})")
    ax.set_title("Quantile coverage (q10–q90)")
    plt.tight_layout(); plt.savefig(out_path, dpi=120); plt.close()

def plot_forecast_examples(pred_df, ret_panel, price_panel, tickers, quantiles, out_path, n_examples=6):
    median_q = str(quantiles[len(quantiles)//2])
    lo_q, hi_q = str(quantiles[0]), str(quantiles[-1])
    windows = pred_df["window"].unique()
    pick = np.linspace(0, len(windows)-1, n_examples).astype(int)
    fig, axes = plt.subplots(2, 3, figsize=(15, 7))
    for ax, w in zip(axes.flat, [windows[i] for i in pick]):
        slab = pred_df[pred_df["window"]==w]
        tic = tickers[0]
        s = slab[slab["id"]==tic].sort_values("timestamp")
        if s.empty: continue
        fut_idx = pd.DatetimeIndex(s["timestamp"].values)
        last_p = price_panel[tic].iloc[price_panel.index.get_loc(fut_idx[0]) - 1]
        p_med = last_p * np.exp(np.cumsum(s[median_q].values))
        p_lo  = last_p * np.exp(np.cumsum(s[lo_q].values))
        p_hi  = last_p * np.exp(np.cumsum(s[hi_q].values))
        truth = price_panel[tic].loc[fut_idx]
        ax.plot(fut_idx, truth.values, "g.-", label="actual")
        ax.plot(fut_idx, p_med, "r.-", label="median")
        ax.fill_between(fut_idx, p_lo, p_hi, color="red", alpha=0.15)
        ax.set_title(f"{tic}  window={w}"); ax.legend(fontsize=7); ax.grid(alpha=0.3)
    plt.tight_layout(); plt.savefig(out_path, dpi=120); plt.close()


## 12. Stage orchestrator

`run_stage(cfg)` is the only thing `runner.ipynb` calls per stage. It snapshots the config, prefetches missing data (idempotent), runs walk-forward, computes metrics + baselines, writes plots, dumps a one-line summary.

In [ ]:
def run_stage(cfg_path: str, cache_dir: Optional[str] = None, pipeline=None):
    cfg = load_config(cfg_path)
    out_dir = Path(cfg["output_dir"]) / cfg["stage_id"]
    out_dir.mkdir(parents=True, exist_ok=True)
    save_config_snapshot(cfg, str(out_dir))
    cache_dir = cache_dir or resolve_cache_dir(cfg)
    # 1. prefetch (idempotent)
    manifest = build_prefetch_manifest([cfg])
    prefetch_all(manifest, cache_dir)
    # 2. cache-only load
    prices, indexes, futures = load_stage_inputs(cfg, cache_dir)
    price_panel, ret_panel, cov_panel = assemble_panels(cfg, prices, indexes, futures)
    print(f"panels: price={price_panel.shape} ret={ret_panel.shape} cov={cov_panel.shape}")
    # 3. pipeline
    if pipeline is None:
        from chronos import Chronos2Pipeline
        pipeline = Chronos2Pipeline.from_pretrained("amazon/chronos-2", device_map=DEVICE, torch_dtype=DTYPE)
    # 4. walk-forward
    preds = run_walk_forward(pipeline, cfg, ret_panel, cov_panel, str(out_dir))
    # 5. baselines
    anchors = walk_forward_anchors(ret_panel.index, cfg["context_len"], cfg["horizon"],
                                   shift=cfg["walk_forward"]["shift"], max_windows=cfg["walk_forward"].get("max_windows"))
    base_preds = {}
    for b in cfg["baselines"]:
        base_preds[b] = baseline_predictions(b, ret_panel, cfg["tickers"], anchors, cfg["context_len"], cfg["horizon"])
    # 6. metrics
    per_cell = per_cell_metrics(preds, ret_panel, cfg["tickers"], cfg["eval_horizons"], cfg["quantiles"])
    per_cell.to_csv(out_dir / "metrics.csv", index=False)
    agg = aggregate_metrics(per_cell); agg.to_csv(out_dir / "metrics_aggregate.csv", index=False)
    base_rows = []
    for b, bp in base_preds.items():
        m = per_cell_metrics(bp, ret_panel, cfg["tickers"], cfg["eval_horizons"], cfg["quantiles"])
        m["baseline"] = b; base_rows.append(m)
    if base_rows:
        pd.concat(base_rows, ignore_index=True).to_csv(out_dir / "metrics_baselines.csv", index=False)
    # 7. plots
    plots = out_dir / "plots"; plots.mkdir(exist_ok=True)
    plot_da_heatmap(per_cell, plots / "da_heatmap.png", title=cfg["stage_id"])
    if "last" in base_preds:
        per_b = per_cell_metrics(base_preds["last"], ret_panel, cfg["tickers"], cfg["eval_horizons"], cfg["quantiles"])
        plot_da_vs_baseline(per_cell, per_b, "last", plots / "da_vs_last.png")
    plot_corr_hist(preds, ret_panel, cfg["eval_horizons"], cfg["quantiles"], plots / "corr_hist.png")
    plot_amplitude(per_cell, plots / "amplitude.png")
    plot_coverage(per_cell, plots / "coverage.png")
    plot_forecast_examples(preds, ret_panel, price_panel, cfg["tickers"], cfg["quantiles"], plots / "examples.png")
    # 8. summary
    primary = per_cell[per_cell["horizon"].isin(cfg["primary_horizons"])]
    summary = dict(
        stage_id=cfg["stage_id"],
        n_windows=int(preds["window"].nunique()) if not preds.empty else 0,
        mean_da_primary=float(primary["da"].mean()) if not primary.empty else None,
        median_da_primary=float(primary["da"].median()) if not primary.empty else None,
        cells_signif_05=int((primary["p_binom_bh"]<0.05).sum()) if not primary.empty else 0,
        mean_pearson_primary=float(primary["pearson"].mean()) if not primary.empty else None,
        mean_coverage_primary=float(primary["coverage"].mean()) if not primary.empty else None,
    )
    with open(out_dir / "summary.json", "w") as f: json.dump(summary, f, indent=2)
    print("SUMMARY:", json.dumps(summary, indent=2))
    return summary


## Usage in `runner.ipynb`

```python
# Cell 1: import everything above (re-paste sections 1, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12)
# Cell 2:
CONFIG_PATH = "configs/stage_1_daily.yaml"
summary = run_stage(CONFIG_PATH)
```

To prefetch all stages at once (do this first on a fresh Drive mount):

```python
all_cfgs = [load_config(p) for p in sorted(Path("configs").glob("stage_*.yaml"))]
cache_dir = resolve_cache_dir(all_cfgs[0])
prefetch_all(build_prefetch_manifest(all_cfgs), cache_dir)
```